In [1]:
"""
stage5b_lgbm_values.py -- Stage 5b: Stage 5 grammar features + causal z-scored
oscillator/leg VALUE features read at t-1. Measures what value shape adds over
pure event timing.

Diff to stage5b_lgbm_values.py. Adds L1/L2 lags of every z-value feature. Note the lag is in the already-t−1-shifted, 
session-local array, so lag-1 here means t−2 relative to the target bar — no leak, 
and it resets at session start because the shift-and-stack happens inside the per-session loop.

RULES
  S5b.1  Additive: X_5b = [ Stage-5 grammar | value block ]. delta vs the Stage 5
         anchor (0.27402) is the marginal contribution of values alone.
  S5b.2  Values are snapshots at t-1 (not events): jmaD1, jmaD2, tick d1/d2 signed
         by leg direction (opposing = positive), their magnitudes, and current leg
         amplitude. Signing by leg_dir makes "opposing momentum" one monotone axis.
  S5b.3  Causal expanding z-score per session: at bar t, standardize using mean/std
         over that session's bars 0..t-1 only. NO whole-session stats (that leaks
         end-of-day variance into the morning). z = 0 until k >= ZWARM.
         This is the batch equivalent of the online Welford recursion in prod;
         _welford_check asserts the two agree.
  S5b.4  Value columns come from raw (jmaD1/jmaD2/tickJmaD1/tickJmaD2/JMA) joined to
         the Featurizer sessions by timestamp. leg_dir and leg_start come from bars.
  S5b.5  Warm-up: value features masked (row dropped) for local bar < ZWARM, on top
         of the Stage-0 warm mask. Prevents training on the unstable-variance region
         that prod emits as 0.
"""

import json
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from scipy.signal import lfilter
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, roc_auc_score
#from stage5_lgbm import (build_lgbm_features, build_meta, LGBM_PARAMS, NUM_ROUNDS, EARLY_STOP)

TAUS = (2.0, 6.0, 18.0)

LGBM_PARAMS = dict(
    objective="binary",
    metric="binary_logloss",
    learning_rate=0.05,
    num_leaves=127,
    min_data_in_leaf=1000,
    feature_fraction=0.9,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l2=1.0,
    num_threads=16,
    verbosity=-1,
)
NUM_ROUNDS = 4000
EARLY_STOP = 200

ZWARM = 20                                                                    # S5b.3
STAGE5_ANCHOR = 0.27402

BODY_OPEN_COL = "rawOpen"        # set per run (HA or raw column names)
BODY_CLOSE_COL = "rawLast"
BODY_TAG = "raw"               # label only, lands in feature names
BODY_SIGNED = True
BODY_MAG = True

RANGE_COL_HIGH = "rawHigh"
RANGE_COL_LOW = "rawLow"
RANGE_TAG = "raw"
ADD_RANGE = True

RAW_VALUE_COLS = ["jmaD1", "jmaD2", "tickJmaD1", "tickJmaD2"]                  # signed
RAW_MAG_COLS = ["jmaD1", "jmaD2", "tickJmaD1", "tickJmaD2"]                    # |.|

RAW_VALUE_COLS = ["jmaD1", "jmaD2", "tickJmaD1", "tickJmaD2"]                  # signed
RAW_MAG_COLS = ["jmaD1", "jmaD2", "tickJmaD1", "tickJmaD2"]                    # |.|
VALUE_LAGS = (1,2,3,4,5,6,7)                                                            # 5c: extra t-2, t-3


In [2]:
def _expanding_z(x, warm):
    """Causal expanding z of x[0..n-1] using stats over 0..t-1. z[t]=0 for t<warm."""
    n = len(x)
    csum = np.concatenate(([0.0], np.cumsum(x)))
    csq = np.concatenate(([0.0], np.cumsum(x * x)))
    k = np.arange(n)                                    # count of bars strictly before t
    with np.errstate(invalid="ignore", divide="ignore"):
        mean = np.where(k > 0, csum[:-1] / np.maximum(k, 1), 0.0)
        var = np.where(k > 1, (csq[:-1] - k * mean * mean) / np.maximum(k - 1, 1), 0.0)
        std = np.sqrt(np.maximum(var, 0.0))
        z = np.where((k >= warm) & (std > 0), (x - mean) / std, 0.0)
    return z


def _welford_check(x, warm):
    """Scalar Welford replay of _expanding_z; returns max abs diff (prod parity)."""
    z_vec = _expanding_z(x, warm)
    k = 0
    mean = 0.0
    M2 = 0.0
    z_ref = np.zeros(len(x))
    for t in range(len(x)):
        if k >= warm and M2 > 0:
            z_ref[t] = (x[t] - mean) / np.sqrt(M2 / (k - 1))
        k += 1
        d = x[t] - mean
        mean += d / k
        M2 += d * (x[t] - mean)
    return float(np.max(np.abs(z_vec - z_ref)))


def build_value_features(fz, raw, date_from=None, date_to=None):
    """Value block, row-aligned with build_lgbm_features over the same range."""
    rv = raw.set_index("timestamp")
    blocks = []


    names = ([f"z_{c}_signed" for c in RAW_VALUE_COLS]
                 + [f"z_{c}_mag" for c in RAW_MAG_COLS]
                 + ["z_leg_amp"]
                 + ([f"z_body_{BODY_TAG}_signed"] if BODY_SIGNED else [])
                 + ([f"z_body_{BODY_TAG}_mag"] if BODY_MAG else [])
                 + ([f"z_range_{RANGE_TAG}", f"z_wick_{RANGE_TAG}"] if ADD_RANGE else []))
   
    names = names + [f"{nm}_lag{L}" for L in VALUE_LAGS for nm in names]    

    for S in fz._selected(date_from, date_to):
        n = S["n"]
        ts = pd.DatetimeIndex(S["timestamp"])
        r = rv.reindex(ts)                                                    # S5b.4
        leg_dir = S["leg_dir"]
        leg_start_val = S["leg_start_jma"]

        feats = []
        for c in RAW_VALUE_COLS:                                              # signed
            v = r[c].to_numpy(np.float64) * leg_dir                           # S5b.2
            feats.append(_expanding_z(v, ZWARM))
        for c in RAW_MAG_COLS:                                                # magnitude
            v = np.abs(r[c].to_numpy(np.float64))
            feats.append(_expanding_z(v, ZWARM))

        amp = np.abs(r["JMA"].to_numpy(np.float64) - leg_start_val)
        feats.append(_expanding_z(amp, ZWARM))
        bo = r[BODY_OPEN_COL].to_numpy(np.float64)
        bc = r[BODY_CLOSE_COL].to_numpy(np.float64)
        if BODY_SIGNED:
            feats.append(_expanding_z((bc - bo) * leg_dir, ZWARM))            # confirming-positive
        if BODY_MAG:
            feats.append(_expanding_z(np.abs(bc - bo), ZWARM))
        if ADD_RANGE:
            rng = (r[RANGE_COL_HIGH].to_numpy(np.float64)
                   - r[RANGE_COL_LOW].to_numpy(np.float64))
            feats.append(_expanding_z(rng, ZWARM))
            feats.append(_expanding_z(rng - np.abs(bc - bo), ZWARM))          # wick = range - |body|

        M = np.stack(feats, 1)                                               # (n, F)
        # shift to t-1 read, then select non-warm rows (must match build_meta order)
        M = np.concatenate([np.zeros((1, M.shape[1])), M[:-1]], 0)            # S5b.2 (t-1)
        # 5c: append lagged copies of the t-1 value block (session-local, zero-padded)
        lagged = [M]
        for L in VALUE_LAGS:
            ML = np.concatenate([np.zeros((L, M.shape[1])), M[:-L]], 0)
            lagged.append(ML)
        M = np.concatenate(lagged, 1)
        t = np.nonzero(~S["warm"])[0]
        Mt = M[t]
        Mt[(t < ZWARM + max(VALUE_LAGS))] = 0.0                               # S5b.5 (+lag warm)
        blocks.append(Mt.astype(np.float32))

    return np.concatenate(blocks), names


def _augment_featurizer(fz, bars):
    """Attach leg_dir and leg_start JMA level per session onto fz.sessions,
    aligned to each session's bar order. Needed by build_value_features."""
    bg = dict(tuple(bars.sort_values("bar_index").groupby("date")))
    for S in fz.sessions:
        g = bg[S["sess"]]
        S["leg_dir"] = g["jma_leg_dir"].to_numpy(np.float64)
        ls = g["bar_index"].to_numpy() - g["bar_index"].to_numpy()[0]
        jma = g["jma"].to_numpy(np.float64)
        # leg_start JMA level: value of jma at the bar that started the current leg
        tgt = g["is_target"].to_numpy()
        starts = np.where(tgt)[0]
        leg_id = np.cumsum(tgt)
        start_idx = np.concatenate(([0], starts))[leg_id]
        S["leg_start_jma"] = jma[start_idx]


def build_X5b(fz, raw, date_from=None, date_to=None):
    Xg, gnames = build_lgbm_features(fz, date_from, date_to)
    Xv, vnames = build_value_features(fz, raw, date_from, date_to)
    assert len(Xg) == len(Xv), (len(Xg), len(Xv))
    return np.hstack([Xg, Xv]), gnames + vnames

In [3]:
# ---------------------------------------------------------------- features

def build_lgbm_features(fz, date_from=None, date_to=None):
    blocks = []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        n = S["n"]
        cols = []
        for c in fz.classes:
            P = S["P"][c]
            ind = np.diff(P).astype(np.float64)            # 0/1 event indicator per bar
            occ = np.where(ind > 0, np.arange(n), -1)
            last = np.maximum.accumulate(occ)
            lastm1 = np.concatenate(([-1], last[:-1]))     # last event <= t-1   S5.2
            bsince = np.where(lastm1 >= 0, np.arange(n) - lastm1, np.arange(n) + 1)
            cols.append(bsince[t])
            x = np.concatenate(([0.0], ind[:-1]))          # events shifted to <= t-1
            for tau in TAUS:
                a = np.exp(-1.0 / tau)
                s = lfilter([a], [1.0, -a], x)             # s_t = a*(s_{t-1} + ind_{t-1})
                cols.append(s[t])
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)           # S5.3
        age = np.where(lt >= 0, t - lt, t + 1)
        cols.append(age)
        cols.append(S["tod"][t].astype(np.float64))
        blocks.append(np.stack(cols, 1).astype(np.float32))

    names = []
    for (s, c) in fz.classes:
        names.append(f"{s}|{c}|bsince")
        names += [f"{s}|{c}|ewm{tau:g}" for tau in TAUS]
    names += ["age", "tod"]
    return np.concatenate(blocks), names


def build_meta(fz, date_from=None, date_to=None):
    """Row-aligned with build_lgbm_features (S5.1)."""
    bi, ts, tg, dt = [], [], [], []
    for S in fz._selected(date_from, date_to):
        t = np.nonzero(~S["warm"])[0]
        bi.append(S["bar_index"][t])
        ts.append(S["timestamp"][t])
        tg.append(S["tgt"][t])
        dt.append(np.full(len(t), str(S["sess"])))
    return pd.DataFrame({"bar_index": np.concatenate(bi),
                         "timestamp": np.concatenate(ts),
                         "is_target": np.concatenate(tg),
                         "date": np.concatenate(dt)})

In [4]:
# ---------------------------------------------------------------- train / eval

def train_lgbm(fz, train_start, train_end, valid_from,
               params=LGBM_PARAMS, num_rounds=NUM_ROUNDS, early_stop=EARLY_STOP):
    X, names = build_lgbm_features(fz, train_start, train_end)
    meta = build_meta(fz, train_start, train_end)
    y = meta["is_target"].to_numpy().astype(np.int8)

    va = (meta["date"] >= valid_from).to_numpy()                               # S5.5
    tr = ~va
    dtr = lgb.Dataset(X[tr], label=y[tr], feature_name=names)
    dva = lgb.Dataset(X[va], label=y[va], reference=dtr)

    booster = lgb.train(params, dtr, num_boost_round=num_rounds,
                        valid_sets=[dva], valid_names=["valid"],
                        callbacks=[lgb.early_stopping(early_stop, verbose=False),
                                   lgb.log_evaluation(200)])

    p_va = booster.predict(X[va], num_iteration=booster.best_iteration)
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_va, y[va])            # S5.6

    print(json.dumps(dict(
        n_train=int(tr.sum()), n_valid=int(va.sum()),
        best_iteration=int(booster.best_iteration),
        valid_logloss_raw=float(log_loss(y[va], p_va)),
        valid_logloss_cal=float(log_loss(y[va], iso.predict(p_va)))), indent=2))

    imp = pd.DataFrame({"feature": names,
                        "gain": booster.feature_importance("gain")}
                       ).sort_values("gain", ascending=False)
    print(imp.head(15).to_string(index=False))

    return dict(booster=booster, iso=iso, feature_names=names,
                train_start=train_start, train_end=train_end,
                valid_from=valid_from, importance=imp)


def evaluate_lgbm(fz, model, start, end=None, stage2_anchor=0.30206):
    X, _ = build_lgbm_features(fz, start, end)
    meta = build_meta(fz, start, end)
    y = meta["is_target"].to_numpy().astype(np.int8)

    p = model["booster"].predict(X, num_iteration=model["booster"].best_iteration)
    p_cal = model["iso"].predict(p)

    ll_raw = log_loss(y, p)
    ll_cal = log_loss(y, p_cal)
    ll_const = log_loss(y, np.full_like(p, y.mean(), dtype=np.float64))
    print(json.dumps(dict(
        n_rows=int(len(y)),
        holdout_logloss_raw=float(ll_raw),
        holdout_logloss_cal=float(ll_cal),
        holdout_logloss_const=float(ll_const),
        auc=float(roc_auc_score(y, p)),
        stage2_anchor=stage2_anchor,
        delta_vs_stage2=float(ll_cal - stage2_anchor)), indent=2))

    out = meta[["bar_index", "timestamp", "is_target"]].copy()                 # S5.7
    out["p"] = p.astype(np.float32)
    out["p_cal"] = p_cal.astype(np.float32)

    tbl = out.assign(bin=pd.qcut(out["p_cal"], 10, duplicates="drop")).groupby(
        "bin", observed=True).agg(mean_p=("p_cal", "mean"),
                                  realized=("is_target", "mean"),
                                  n=("p_cal", "size"))
    print(tbl.to_string())
    return out

In [5]:
# ---------------------------------------------------------------- train / eval

def train_5b(fz, raw, train_start, train_end, valid_from,
             params=LGBM_PARAMS, num_rounds=NUM_ROUNDS, early_stop=EARLY_STOP):
    X, names = build_X5b(fz, raw, train_start, train_end)
    meta = build_meta(fz, train_start, train_end)
    y = meta["is_target"].to_numpy().astype(np.int8)

    va = (meta["date"] >= valid_from).to_numpy()
    tr = ~va
    dtr = lgb.Dataset(X[tr], label=y[tr], feature_name=names)
    dva = lgb.Dataset(X[va], label=y[va], reference=dtr)
    booster = lgb.train(params, dtr, num_boost_round=num_rounds,
                        valid_sets=[dva], valid_names=["valid"],
                        callbacks=[lgb.early_stopping(early_stop, verbose=False),
                                   lgb.log_evaluation(200)])
    p_va = booster.predict(X[va], num_iteration=booster.best_iteration)
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_va, y[va])

    print(json.dumps(dict(n_train=int(tr.sum()), n_valid=int(va.sum()),
                          best_iteration=int(booster.best_iteration),
                          valid_logloss_cal=float(log_loss(y[va], iso.predict(p_va)))),
                     indent=2))
    imp = pd.DataFrame({"feature": names,
                        "gain": booster.feature_importance("gain")}
                       ).sort_values("gain", ascending=False)
    print(imp.head(20).to_string(index=False))
    return dict(booster=booster, iso=iso, feature_names=names, importance=imp)


def evaluate_5b(fz, raw, model, start, end=None, anchor=STAGE5_ANCHOR):
    X, _ = build_X5b(fz, raw, start, end)
    meta = build_meta(fz, start, end)
    y = meta["is_target"].to_numpy().astype(np.int8)
    p = model["booster"].predict(X, num_iteration=model["booster"].best_iteration)
    p_cal = model["iso"].predict(p)

    ll_cal = log_loss(y, p_cal)
    ll_const = log_loss(y, np.full_like(p, y.mean(), dtype=np.float64))
    print(json.dumps(dict(n_rows=int(len(y)),
                          holdout_logloss_cal=float(ll_cal),
                          holdout_logloss_const=float(ll_const),
                          skill=float(1 - ll_cal / ll_const),
                          auc=float(roc_auc_score(y, p)),
                          stage5_anchor=anchor,
                          delta_vs_stage5=float(ll_cal - anchor)), indent=2))

    out = meta[["bar_index", "timestamp", "is_target"]].copy()
    out["p"] = p.astype(np.float32)
    out["p_cal"] = p_cal.astype(np.float32)
    tbl = out.assign(bin=pd.qcut(out["p_cal"], 10, duplicates="drop")).groupby(
        "bin", observed=True).agg(mean_p=("p_cal", "mean"),
                                  realized=("is_target", "mean"), n=("p_cal", "size"))
    print(tbl.to_string())
    return out

In [6]:
def _make_classes():
    cls = []
    for s in STREAMS:
        cls.append((s, "opp"))
        cls.append((s, "conf"))
    cls.append(("MNQ_JMA_SELF", "all"))
    return cls

def _bins_from_edges(edges):
    bins, lo = [], 1
    for e in edges:
        bins.append((lo, int(e)))
        lo = int(e) + 1
    return bins

def _age_bins_from_edges(edges):
    b = _bins_from_edges(edges)
    b.append((int(edges[-1]) + 1, 10 ** 9))
    return b

class Featurizer:
    def __init__(self, bars, events,
                 bin_edges=(1, 2, 3, 5, 8, 13, 21, 34),
                 age_edges=(1, 2, 3, 5, 8, 13, 21, 34, 55, 89)):
        self.bin_edges = list(bin_edges)
        self.age_edges = list(age_edges)
        self.bins = _bins_from_edges(bin_edges)
        self.age_bins = _age_bins_from_edges(age_edges)
        self.classes = _make_classes()
        self.feature_names = (
            [f"{s}|{c}|lag{lo}_{hi}" for (s, c) in self.classes for (lo, hi) in self.bins]
            + [f"age|{lo}_{hi}" for (lo, hi) in self.age_bins]
            + [f"tod|{k}" for k in range(N_TOD)]
        )
        self.n_feat = len(self.feature_names)

        ev = events[events["stream"].isin([c[0] for c in self.classes])]
        ev = ev[~((ev["stream"] != "MNQ_JMA_SELF") & (ev["opposing"] == 0))]   # S2.6
        evg = dict(tuple(ev.groupby("date")))

        self.sessions = []
        bars = bars.sort_values("bar_index").reset_index(drop=True)
        for sess, g in bars.groupby("date", sort=True):
            n = len(g)
            first = int(g["bar_index"].iloc[0])
            tgt = g["is_target"].to_numpy()
            warm = g["warm"].to_numpy()
            ts = pd.DatetimeIndex(g["timestamp"])
            mins = ts.hour.to_numpy() * 60 + ts.minute.to_numpy() - TOD_START_MIN   # S2.10
            tod = np.clip(mins // TOD_BIN_MIN, 0, N_TOD - 1).astype(np.int16)
            lt_incl = np.maximum.accumulate(np.where(tgt, np.arange(n), -1))
            P = {}
            e = evg.get(sess)
            for (s, c) in self.classes:
                if e is None:
                    loc = np.empty(0, np.int64)
                else:
                    if s == "MNQ_JMA_SELF":
                        sub = e[e["stream"] == s]
                    else:
                        want = 1 if c == "opp" else -1
                        sub = e[(e["stream"] == s) & (e["opposing"] == want)]
                    loc = sub["event_bar"].to_numpy() - first
                ind = np.zeros(n, np.int32)
                ind[loc] = 1
                P[(s, c)] = np.concatenate(([0], np.cumsum(ind)))              # P[i] = count pos < i
            self.sessions.append(dict(
                sess=sess, first=first, n=n, tgt=tgt, warm=warm, tod=tod,
                lt_incl=lt_incl, P=P,
                bar_index=g["bar_index"].to_numpy(),
                timestamp=g["timestamp"].to_numpy()))

    def _fill(self, S, t, out):
        n = S["n"]
        col = 0
        for c in self.classes:
            P = S["P"][c]
            for (lo, hi) in self.bins:                                         # S2.1, S2.2
                b = np.clip(t - lo + 1, 0, n)
                a = np.clip(t - hi, 0, n)
                out[:, col] = P[b] - P[a]
                col += 1
        lt = np.where(t > 0, S["lt_incl"][np.maximum(t - 1, 0)], -1)           # S2.3
        age = np.where(lt >= 0, t - lt, t + 1)
        for (lo, hi) in self.age_bins:
            out[:, col] = (age >= lo) & (age <= hi)
            col += 1
        tod = S["tod"][t]
        for k in range(N_TOD):
            out[:, col] = tod == k
            col += 1

    def _fill_frozen(self, S, q, h, out):
        """S2.5: features at t = q+h using only events/targets with pos <= q."""
        n = S["n"]
        t = q + h
        cap = q + 1
        col = 0
        for c in self.classes:
            P = S["P"][c]
            for (lo, hi) in self.bins:
                b = np.clip(np.minimum(t - lo + 1, cap), 0, n)
                a = np.clip(np.minimum(t - hi, cap), 0, n)
                out[:, col] = P[b] - P[a]
                col += 1
        lt = S["lt_incl"][q]
        age = np.where(lt >= 0, t - lt, t + 1)
        for (lo, hi) in self.age_bins:
            out[:, col] = (age >= lo) & (age <= hi)
            col += 1
        tod = S["tod"][np.minimum(t, n - 1)]
        for k in range(N_TOD):
            out[:, col] = tod == k
            col += 1

    def _selected(self, date_from, date_to):
        for S in self.sessions:
            d = str(S["sess"])
            if date_from is not None and d < date_from:
                continue
            if date_to is not None and d > date_to:
                continue
            yield S

    def build(self, date_from=None, date_to=None):
        sel = []
        total = 0
        for S in self._selected(date_from, date_to):
            t = np.nonzero(~S["warm"])[0]                                      # S2.7
            sel.append((S, t))
            total += len(t)
        X = np.zeros((total, self.n_feat), np.float32)
        y = np.zeros(total, np.int8)
        bar_index = np.zeros(total, np.int64)
        sess_id = np.zeros(total, np.int32)
        timestamp = np.zeros(total, "datetime64[ns]")
        ofs = 0
        for sid, (S, t) in enumerate(sel):
            m = len(t)
            self._fill(S, t, X[ofs:ofs + m])
            y[ofs:ofs + m] = S["tgt"][t]
            bar_index[ofs:ofs + m] = S["bar_index"][t]
            sess_id[ofs:ofs + m] = sid
            timestamp[ofs:ofs + m] = S["timestamp"][t]
            ofs += m
        meta = pd.DataFrame({"bar_index": bar_index, "timestamp": timestamp,
                             "sess_id": sess_id, "is_target": y.astype(bool)})
        return X, y, meta

In [7]:
# ---------------------------------------------------------------- usage
# ---- CONFIG ----
FRAME = 6
BARS_PATH = f"data/stage-0/bars_{FRAME}s.parquet"
EVENTS_PATH = f"data/stage-0/events_{FRAME}s.parquet"
RAW_PATH = "data/mnq-tick-oscillator-6sec.pqt"
OUT_DIR = "data/stage-5"
VALID_FROM = "2024-07-01"
TRAIN_END = "2024-12-31"
TEST_FROM = "2025-01-01"

RAW_OHLC_FILE = f'data/mnq-ohlc-raw-6sec.pqt'

# ----------------
TOD_START_MIN = 570      # first session minute (9:30)
TOD_BIN_MIN = 30
N_TOD = 13               # bins covering the session 6.5hrs / 30 min

STREAMS = ["MNQ_D1", "MNQ_D2", "TICK_D1", "TICK_D2"]
# ----------------

###########################################
# USING RAW OHLC instead of HA
###########################################
bars = pd.read_parquet(BARS_PATH)
events = pd.read_parquet(EVENTS_PATH)
raw = pd.read_parquet(RAW_PATH)
raw = raw[raw["timestamp"].dt.time >= pd.Timestamp("09:30").time()]
ohlc = pd.read_parquet(RAW_OHLC_FILE)
ohlc = ohlc[ohlc["timestamp"].dt.time >= pd.Timestamp("09:30").time()]
ohlc = ohlc.rename(columns={"Open": "rawOpen", "High": "rawHigh", "Low": "rawLow", "Last": "rawLast"})

raw = raw.merge(ohlc, on="timestamp", how="left")
assert raw[["rawOpen", "rawLast"]].notna().all().all(), "raw OHLC has gaps vs oscillator timestamps"

print(ohlc.info())
print(ohlc.head())

<class 'pandas.DataFrame'>
RangeIndex: 4461888 entries, 0 to 4461887
Data columns (total 6 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   rawOpen    float64       
 2   rawHigh    float64       
 3   rawLow     float64       
 4   rawLast    float64       
 5   date       datetime64[us]
dtypes: datetime64[us](2), float64(4)
memory usage: 204.2 MB
None
            timestamp   rawOpen   rawHigh    rawLow   rawLast       date
0 2022-01-03 09:30:00  16382.75  16395.25  16378.50  16395.25 2022-01-03
1 2022-01-03 09:30:06  16395.25  16397.25  16390.50  16395.75 2022-01-03
2 2022-01-03 09:30:12  16395.50  16402.25  16389.50  16401.75 2022-01-03
3 2022-01-03 09:30:18  16401.75  16409.00  16399.25  16407.25 2022-01-03
4 2022-01-03 09:30:24  16407.50  16413.00  16404.25  16405.25 2022-01-03


In [8]:
fz = Featurizer(bars, events)
_augment_featurizer(fz, bars)

# prod-parity check: vectorized z == scalar Welford on one session
S0 = fz.sessions[len(fz.sessions) // 2]
xchk = (raw.set_index("timestamp").reindex(pd.DatetimeIndex(S0["timestamp"]))
        ["jmaD1"].to_numpy(np.float64))
print("welford max abs diff:", _welford_check(xchk, ZWARM))               # expect ~0

model = train_5b(fz, raw, None, TRAIN_END, VALID_FROM)
pred = evaluate_5b(fz, raw, model, TEST_FROM)

joblib.dump({k: v for k, v in model.items() if k != "importance"},
            f"{OUT_DIR}/lgbm5b_model_{FRAME}s.joblib")
model["importance"].to_csv(f"{OUT_DIR}/lgbm5b_importance_{FRAME}s.csv", index=False)
pred.to_parquet(f"{OUT_DIR}/pred_lgbm5b_{FRAME}s.parquet", index=False)

welford max abs diff: 2.4868995751603507e-14
[200]	valid's binary_logloss: 0.130886
[400]	valid's binary_logloss: 0.127047
[600]	valid's binary_logloss: 0.126063
[800]	valid's binary_logloss: 0.125726
[1000]	valid's binary_logloss: 0.125539
[1200]	valid's binary_logloss: 0.125437
[1400]	valid's binary_logloss: 0.125421
{
  "n_train": 2462897,
  "n_valid": 495290,
  "best_iteration": 1353,
  "valid_logloss_cal": 0.12490532487473781
}
                feature         gain
      z_body_raw_signed 1.653834e+06
         z_jmaD1_signed 1.055276e+06
         z_jmaD2_signed 1.003714e+06
            z_jmaD1_mag 9.073322e+05
 z_body_raw_signed_lag1 8.347783e+05
         z_body_raw_mag 6.647205e+05
       z_jmaD1_mag_lag1 3.937300e+05
    z_body_raw_mag_lag1 2.886399e+05
      MNQ_D1|opp|bsince 2.482181e+05
 z_body_raw_signed_lag2 2.139901e+05
            z_jmaD2_mag 2.011034e+05
    z_jmaD1_signed_lag1 1.794522e+05
       MNQ_D1|conf|ewm2 1.696666e+05
MNQ_JMA_SELF|all|bsince 1.211266e+05
        